In [8]:
import xarray as xr
import numpy as np
import xesmf as xe #this package is very unestable, if error try restart krenel
from dask.diagnostics import ProgressBar
import json


In [9]:
GFW_ds = xr.open_zarr(
    "./resources/GFW/GFW_AIS_Intl-FK_Trwl-Jgr_2012-2025.zarr",
    consolidated=True,
    decode_timedelta=True
)

In [10]:
GFW_ds

<xarray.Dataset> Size: 324MB
Dimensions:   (flag: 5, geartype: 2, time: 168, lat: 171, lon: 141)
Coordinates:
  * flag      (flag) object 40B 'CHN' 'ESP' 'FLK' 'KOR' 'TWN'
  * geartype  (geartype) object 16B 'SQUID_JIGGER' 'TRAWLERS'
  * lat       (lat) float64 1kB -57.0 -56.9 -56.8 -56.7 ... -40.2 -40.1 -40.0
  * lon       (lon) float64 1kB -66.0 -65.9 -65.8 -65.7 ... -52.2 -52.1 -52.0
  * time      (time) datetime64[ns] 1kB 2012-01-01 2012-02-01 ... 2025-12-01
Data variables:
    hours     (time, lat, lon, flag, geartype) float64 324MB dask.array<chunksize=(12, 171, 141, 1, 1), meta=np.ndarray>
Attributes:
    created_by:    Ruben Barriuso
    date_created:  2025-12-29
    description:   Monthly fishing hours per grid cell in the SW Atlantic, \n...
    resolution:    0.1x0.1 degree
    source:        Global Fishing Watch (GFW)
    title:         SW Atlantic GFW AIS Fishing Effort Data (2012-2024)

In [11]:
with open("./data/bbox.json", "r") as f:
    bbox = json.load(f)

min_lon, min_lat, max_lon, max_lat = [
    bbox["min_lon"],
    bbox["min_lat"],
    bbox["max_lon"],
    bbox["max_lat"]
]
print(min_lon, min_lat, max_lon, max_lat)


res = 0.125
new_lons = np.arange(min_lon, max_lon + res, res)
new_lats = np.arange(min_lat, max_lat + res, res)

# Create target grid for from our study bbox with resolution of 0.125
ds_tgt = xr.Dataset({
    'lat': (['lat'], new_lats),
    'lon': (['lon'], new_lons)})

regridder = xe.Regridder(GFW_ds, ds_tgt, 'conservative') #regrid from the 0.1 resolution to the 0.125 resolution of our study 
with ProgressBar():
    GFW_regridded = regridder(GFW_ds).compute() #compute (as is a dask zarray)


-66.0 -57.0 -52.0 -40.0
[########################################] | 100% Completed | 1.34 sms


In [12]:
#comparision of the original and the regrided grids
print("original size", GFW_ds.sizes)
print("regrided size", GFW_regridded.sizes)
lat_diff = np.diff(GFW_regridded.lat)
lon_diff = np.diff(GFW_regridded.lon)
print("Regrided spacing:")
print("Lat spacing:", np.unique(lat_diff), "Lon spacing:", np.unique(lon_diff))
print("Original bounds:")
print(GFW_ds.lat.min().item(), GFW_ds.lat.max().item())
print(GFW_ds.lon.min().item(), GFW_ds.lon.max().item())
print("Regrided bounds:")
print(GFW_regridded.lat.min().item(), GFW_regridded.lat.max().item())
print(GFW_regridded.lon.min().item(), GFW_regridded.lon.max().item())

original size Frozen({'flag': 5, 'geartype': 2, 'time': 168, 'lat': 171, 'lon': 141})
regrided size Frozen({'time': 168, 'flag': 5, 'geartype': 2, 'lat': 137, 'lon': 113})
Regrided spacing:
Lat spacing: [0.125] Lon spacing: [0.125]
Original bounds:
-57.0 -39.99999999999976
-66.0 -52.000000000000796
Regrided bounds:
-57.0 -40.0
-66.0 -52.0


In [13]:
#Adding metadata and description to this zarr dataset
GFW_regridded.attrs['title'] = "Regrided SW Atlantic GFW AIS Fishing Effort Data (2012-2024)"
GFW_regridded.attrs["description"] = (
    """Monthly fishing hours per grid cell in the SW Atlantic, 
    aligned to 0.125° lat/lon grid
    only data from trawlers and squid jiggers vessels included,
    only data within the available fishing area is included which comprises
    international waters and claimed Falkland Islands EEZ."""
)

GFW_regridded.attrs["source"] = "AIS-based fishing effort dataset"
GFW_regridded.attrs["created_by"] = "Ruben Barriuso"
GFW_regridded.attrs["resolution"] = "0.125x0.125 degree"
GFW_regridded.attrs["source"] = "Global Fishing Watch (GFW)"

GFW_regridded["hours"].attrs["long_name"] = "Fishing effort in AIS hours"
GFW_regridded["hours"].attrs["description"] = (
    "Total monthly fishing effort observed in each grid cell from GFW"
)

In [14]:
for var in GFW_regridded.variables:
    GFW_regridded[var].encoding = {} #this is for elimiating the encoding, causes problems in future analysis
GFW_regridded.to_zarr("./data/processed/GFW_AIS.zarr", 
                      mode='w',
                      consolidated=True)#creates a metadata file for faster access)